# ErgoPose Risk Classifier — Model Training

This notebook is the **third stage** of the *ErgoPose Risk Classifier* project.  
It defines, trains, and evaluates the **Artificial Neural Network (ANN)** for posture classification based on the preprocessed dataset.

### Objectives
- Load the cleaned dataset from `data/processed/`.
- Encode categorical posture labels.
- Split the data into training and testing sets.
- Define and train an ANN model for multi-class classification.
- Save the trained model and scaler to the `models/` directory.

### Input and Output
- **Input:** `data/processed/clean_postural_risk_dataset.csv`  
- **Outputs:**  
  - `models/neural_network.pkl`  
  - `models/scaler.pkl`

In [1]:
"""
Imports the necessary libraries for model definition, training, and evaluation.
"""

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import joblib
import json
from math import sqrt

import torch
from torch import Tensor, nn, optim

In [2]:
"""
Defines paths for processed data and model output directories.
"""

DATA_PATH = Path("../data/processed/clean_postural_risk_dataset.csv")
MODELS_PATH = Path("../models")
MODELS_PATH.mkdir(exist_ok=True)

print(f"Dataset path: {DATA_PATH}")
print(f"Models directory: {MODELS_PATH}")


Dataset path: ../data/processed/clean_postural_risk_dataset.csv
Models directory: ../models


In [3]:
"""
Defines the number of neurons in the hidden layers by the 'Geometric Pyramid Rule'
"""

data = pd.read_csv(DATA_PATH)

X = data.drop(columns=['upperbody_label'])
y = data['upperbody_label']

input_neurons = X.shape[1]
output_neurons = 2 # Binary classification

input_output_neurons = int(sqrt(input_neurons*output_neurons))

print(f"The number of neurons in hidden layers will be: {int(input_output_neurons*0.5)} <= N <= {int(input_output_neurons*2)}")

The number of neurons in hidden layers will be: 5 <= N <= 20


## Model Architecture Proposals - rules
- Hidden layers must have beetwen **5 and 20 neurons total**. If more than 1 hidden layer is implemented, the number of neurons of both layers must **add up** to a number beetwen **5 and 20**.
- Batch size, at this initial stage, must be **default**.
- Activation function **cannot** be tanh.
- Learning rate must be $10^{-2}, 10^{-3}$ or **smaller numbers**.

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [5]:
def run_model(input_dim, hidden_dim_1, hidden_dim_2, output_dim, kf, best_acc, best_model):
    acc_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    X_t: Tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
    
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    y_enc = torch.tensor(y_enc, dtype=torch.float32)
    
    for fold, (train_index, val_index) in enumerate(kf.split(X_t)):
        X_train_t, X_val_t = X_t[train_index].to(device), X_t[val_index].to(device)
        y_train_t, y_val_t = y_enc[train_index].to(device), y_enc[val_index].to(device)
        
        model: MLPModel = MLPModel(input_dim, hidden_dim_1, hidden_dim_2, output_dim).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        
        for epoch in range(300):
            model.train()
            optimizer.zero_grad()
            outputs = model(X_train_t).squeeze()
            loss = criterion(outputs, y_train_t)
            loss.backward()
            optimizer.step()
            # print(f"Epoch {epoch}: loss {loss}")
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t).squeeze()
            y_pred_t = (torch.sigmoid(val_outputs) > 0.5).long()
            y_pred = y_pred_t.cpu().numpy()
            y_true = y_val_t.cpu().numpy()
        
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        if acc > best_acc:
            best_acc = acc
            best_model = model
            
            print(f"New best model found (Acc {best_acc:.4f})")
        
        acc_scores.append(acc)
        precision_scores.append(prec)
        recall_scores.append(rec)
        f1_scores.append(f1)

    results = {
        "acc": np.mean(acc_scores),
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "f1": np.mean(f1_scores)
    }
        
    return results, best_model, best_acc

In [6]:
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim_1, hidden_dim_2, output_dim):
        super(MLPModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Linear(hidden_dim_2, output_dim),
        )

    def forward(self, x):
        return self.net(x)

number_tests = 10
metrics = []
best_acc = 0.0
best_model = None

for test in range(number_tests):
    seed = 42 + test
    
    num_splits = 5
    kf = KFold(
        n_splits=num_splits,
        shuffle=True,
        random_state=seed
    )
    
    # Hyperparameters
    input_dim: int = X.shape[1]
    hidden_dim_1 = 8
    hidden_dim_2 = 10
    output_dim = 1

    results, new_best_model, new_best_acc = run_model(input_dim, hidden_dim_1, hidden_dim_2, output_dim, kf, best_acc, best_model)

    best_acc = new_best_acc
    best_model = new_best_model
    
    metrics.append(results)
    
    print(f"Métricas da execução {test}--------------------------")
    print(f"""
    Acc: {results['acc']}
    Precision: {results['precision']}
    Recall: {results['recall']}
    F1: {results['f1']}
    """)

final_metrics = {
    "Acc": np.mean([m["acc"] for m in metrics]),
    "Precision": np.mean([m["precision"] for m in metrics]),
    "Recall": np.mean([m["recall"] for m in metrics]),
    "F1": np.mean([m["f1"] for m in metrics]),
}
    
print(f"Acurácia: {final_metrics['Acc']:.4f}")
print(f"Precisão: {final_metrics['Precision']:.4f}")
print(f"Revocação: {final_metrics['Recall']:.4f}")
print(f"F1: {final_metrics['F1']:.4f}")

print(f"Best model Accuracy: {best_acc:.4f}")

New best model found (Acc 0.7810)
New best model found (Acc 0.8123)
Métricas da execução 0--------------------------

    Acc: 0.7813961132965139
    Precision: 0.6811842384943636
    Recall: 0.6620176462418312
    F1: 0.6711060546770686
    
Métricas da execução 1--------------------------

    Acc: 0.7841068353647784
    Precision: 0.6750873924971996
    Recall: 0.6934665894397903
    F1: 0.6831474578400101
    
Métricas da execução 2--------------------------

    Acc: 0.7843129913074901
    Precision: 0.6759704188184646
    Recall: 0.6960107678693418
    F1: 0.6847802082012155
    
New best model found (Acc 0.8142)
Métricas da execução 3--------------------------

    Acc: 0.7926619804467511
    Precision: 0.6895893774962681
    Recall: 0.7001980453202139
    F1: 0.6937257858545707
    
New best model found (Acc 0.8319)
Métricas da execução 4--------------------------

    Acc: 0.7891203214900699
    Precision: 0.6827739011461327
    Recall: 0.702794875996886
    F1: 0.690920168070

In [7]:
model_path = f"{MODELS_PATH}/modelo_{round(final_metrics['Acc'], 2)}_{hidden_dim_1}-{hidden_dim_2}.pth"
model_details_path = f"{MODELS_PATH}/modelo_{round(final_metrics['Acc'], 2)}_{hidden_dim_1}-{hidden_dim_2}.json"

model = best_model

torch.save(model, model_path)

model_details = {
    "optimizer": "Adam",
    "funcao_ativacao": "ReLU",
    "num_epochs": 300,
    "hidden_layer_1": hidden_dim_1,
    "hidden_layer_2": hidden_dim_2,
    "learning_rate": 0.0001,
    "acc": round(final_metrics['Acc'], 2),
    "precision": round(final_metrics['Precision'], 2),
    "recall": round(final_metrics['Recall'], 2),
    "f1": round(final_metrics['F1'], 2)
}

with open(model_details_path, 'w') as f:
    json.dump(model_details, f, indent=4)


print(f"Modelo salvo em {model_path}.")
print(f"Detalhes salvos em {model_details_path}")

Modelo salvo em ../models/modelo_0.79_8-10.pth.
Detalhes salvos em ../models/modelo_0.79_8-10.json
